In [0]:
target_catalog_name = dbutils.widgets.get("catalog_name")
silver_schema = dbutils.widgets.get("silver_schema")
bronze_schema = dbutils.widgets.get("bronze_schema")
columns = ["bookings.booking_id",
                             "bookings.check_in",
                             "bookings.check_out",
                             "bookings.status",
                             "bookings.total_amount",
                             "users.user_id",
                             "users.name",
                             "users.country",
                             "users.user_type",
                             "properties.property_id",
                             "properties.title"]
source_name = dbutils.widgets.get("source_name")

In [0]:
bookings_df = spark.table(f"`{target_catalog_name}`.`{bronze_schema}`.`{source_name}_bookings`")
users_df = spark.table(f"`{target_catalog_name}`.`{bronze_schema}`.`{source_name}_users`")
properties_df = spark.table(f"`{target_catalog_name}`.`{bronze_schema}`.`{source_name}_properties`")

main_df = bookings_df.alias("bookings") \
                     .filter("status = 'confirmed'") \
                     .join(users_df.alias("users"), "user_id", "left") \
                     .join(properties_df.alias("properties"), "property_id", "left") \
                     .select(*columns) \
                     .withColumnRenamed("title", "Hotel_name") \
                     .withColumnRenamed("user_id", "guest_id")

# Exceptions
invalid_bookings_df = main_df.filter("""
    booking_id IS NULL
    OR guest_id IS NULL
    OR property_id IS NULL
    OR total_amount < 0
""")

valid_bookings_df = main_df.filter("""
    booking_id IS NOT NULL
    AND guest_id IS NOT NULL
    AND property_id IS NOT NULL
    AND total_amount >= 0
""")

invalid_bookings_df.write.mode("overwrite").saveAsTable(f"`{target_catalog_name}`.`{silver_schema}`.`{source_name}_main_quarantine`")
valid_bookings_df.write.mode("overwrite").saveAsTable(f"`{target_catalog_name}`.`{silver_schema}`.`{source_name}_main`")